<a href="https://colab.research.google.com/github/codewithfifa/CODSOFT_AI/blob/main/Task4_Recommendation_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ----------------- 1. CREATE SAMPLE DATASET -----------------
# Built-in movie dataset with titles, genres, and summaries
movies_data = {
    "title": [
        "The Dark Knight",
        "Batman Begins",
        "The Avengers",
        "Iron Man",
        "Interstellar",
        "Inception",
        "The Notebook",
        "Titanic",
        "Toy Story",
        "Finding Nemo",
        "The Conjuring",
        "A Quiet Place",
    ],
    "genre": [
        "Action Crime Drama Superhero",
        "Action Crime Superhero Vigilante",
        "Action Adventure Sci-Fi Superhero",
        "Action Sci-Fi Superhero Marvel",
        "Sci-Fi Adventure Space Exploration Drama",
        "Sci-Fi Action Thriller Dream Heist",
        "Romance Drama Love Story",
        "Romance Drama Disaster Ship Tragedy",
        "Animation Adventure Comedy Toys Family",
        "Animation Adventure Family Ocean Fish",
        "Horror Mystery Thriller Haunted Ghost",
        "Horror Sci-Fi Thriller Monster Survival",
    ],
    "description": [
        "A superhero battles the Joker to save Gotham City from chaos.",
        "Bruce Wayne trains with martial artists and becomes Batman to fight crime in Gotham.",
        "Superheroes unite to fight an alien invasion led by Loki.",
        "A billionaire engineer creates a high-tech suit of armor to fight terrorists and criminals.",
        "Astronauts travel through a wormhole in space to find a new habitable planet for humanity.",
        "A thief who steals corporate secrets through dream-sharing technology is given a chance at redemption.",
        "A poor young man and a rich young woman fall deeply in love through decades of separation.",
        "A seventeen-year-old aristocrat falls in love with a kind but poor artist aboard the luxurious, ill-fated ship.",
        "A cowboy doll is threatened when a new spaceman figure becomes the top toy in a boy's room.",
        "After his son is captured in the Great Barrier Reef, a timid clownfish sets out on a journey to bring him home.",
        "Paranormal investigators help a family terrorized by a dark presence in their farmhouse.",
        "A family struggles to survive in a post-apocalyptic world inhabited by blind creatures with ultra-sensitive hearing.",
    ],
}

# Create a DataFrame
df = pd.DataFrame(movies_data)

# Combine genres and descriptions into one feature for the AI
df["combined_features"] = df["genre"] + " " + df["description"]

# ----------------- 2. CONTENT-BASED FILTERING -----------------
# Convert text features into numerical vectors using TF-IDF
tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(df["combined_features"])

# Compute cosine similarity between all movies
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)


# ----------------- 3. RECOMMENDATION FUNCTION -----------------
def recommend_movies(movie_title, num_recommendations=3):
    # Find matching movie (case-insensitive)
    matches = df[df["title"].str.lower() == movie_title.lower()]

    if matches.empty:
        return (
            f"❌ Sorry, '{movie_title}' is not in the database.\n"
            f"Available movies: {', '.join(df['title'].tolist())}"
        )

    # Get the index of the movie
    movie_index = matches.index[0]

    # Get similarity scores for all movies with that movie
    sim_scores = list(enumerate(cosine_sim[movie_index]))

    # Sort the movies based on similarity scores (descending)
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Exclude the movie itself (index 0 in sorted list) and take top N
    sim_scores = sim_scores[1 : num_recommendations + 1]

    recommendations = []
    for idx, score in sim_scores:
        recommendations.append(
            (df.iloc[idx]["title"], df.iloc[idx]["genre"], round(score * 100, 1))
        )

    return recommendations


# ----------------- 4. USER INTERACTION LOOP -----------------
def run_recommender():
    print("=" * 60)
    print("        AI MOVIE RECOMMENDATION SYSTEM (CONTENT-BASED)       ")
    print("=" * 60)
    print("Available Movies in Database:")
    for title in df["title"]:
        print(f" • {title}")
    print("-" * 60)
    print("Type a movie name to get recommendations, or 'exit' to quit.\n")

    while True:
        user_movie = input("Enter a movie title: ").strip()

        if user_movie.lower() in ["exit", "quit", "bye"]:
            print("Thank you for using the Recommendation System!")
            break

        if not user_movie:
            continue

        result = recommend_movies(user_movie)

        if isinstance(result, str):
            print(result)
        else:
            print(f"\n🍿 Top Recommendations for '{user_movie}':")
            for i, (title, genre, match_pct) in enumerate(result, 1):
                print(
                    f"  {i}. {title} (Genre: {genre}) — Match: {match_pct}%"
                )
        print("-" * 60)


if __name__ == "__main__":
    run_recommender()


        AI MOVIE RECOMMENDATION SYSTEM (CONTENT-BASED)       
Available Movies in Database:
 • The Dark Knight
 • Batman Begins
 • The Avengers
 • Iron Man
 • Interstellar
 • Inception
 • The Notebook
 • Titanic
 • Toy Story
 • Finding Nemo
 • The Conjuring
 • A Quiet Place
------------------------------------------------------------
Type a movie name to get recommendations, or 'exit' to quit.

Enter a movie title: Inception

🍿 Top Recommendations for 'Inception':
  1. The Avengers (Genre: Action Adventure Sci-Fi Superhero) — Match: 9.8%
  2. A Quiet Place (Genre: Horror Sci-Fi Thriller Monster Survival) — Match: 8.6%
  3. Iron Man (Genre: Action Sci-Fi Superhero Marvel) — Match: 8.3%
------------------------------------------------------------
Enter a movie title: the dark knight

🍿 Top Recommendations for 'the dark knight':
  1. Batman Begins (Genre: Action Crime Superhero Vigilante) — Match: 33.4%
  2. The Avengers (Genre: Action Adventure Sci-Fi Superhero) — Match: 14.7%
  3. Iron 